In [2]:
# Cell 1 - Install & imports, load dataset, inspect
!pip install -q sentence-transformers faiss-cpu

import os
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import faiss   # FAISS used later for fast search

# --------------------------
# Set your dataset path here
# Option A: If you uploaded the CSV to Colab files, use:
# DATA_PATH = "/content/formatted_jobs.csv"
#
# Option B: If file is in your Google Drive, mount drive and set path:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/your_folder/formatted_jobs.csv"
#
# Replace below with your actual path
DATA_PATH = "/kaggle/input/jobs-and-skills-mapping-for-career-analysis/formatted_jobs.csv"

# If default path doesn't exist, show files in /content to help you
if not os.path.exists(DATA_PATH):
    print("WARNING: DATA_PATH does not exist:", DATA_PATH)
    print("Files in /content:")
    print(os.listdir("/content"))
    raise FileNotFoundError("Please upload your CSV to Colab and update DATA_PATH variable in this cell.")

# Load dataset
df = pd.read_csv(DATA_PATH)

# Quick checks
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nSample rows:")
display(df.head(6))

# Basic sanity checks for required columns
required_cols = ["job_title", "Short_description", "Skills_required", "Industry", "Pay_grade"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required column(s): {missing}. Please ensure CSV contains these columns.")
else:
    print("\n✅ All required columns present.")


Dataset Shape: (970, 6)

Columns: ['ID_num', 'job_title', 'Short_description', 'Skills_required', 'Industry', 'Pay_grade']

Dtypes:
 ID_num                int64
job_title            object
Short_description    object
Skills_required      object
Industry             object
Pay_grade            object
dtype: object

Sample rows:


,ID_num,job_title,Short_description,Skills_required,Industry,Pay_grade
0,1,Software Engineer,Develop and maintain web applications using mo...,Problem Solving Logical Reasoning Attention to...,Technology,High paying
1,2,Data Scientist,Analyze large datasets to extract business ins...,Analytical Thinking Pattern Recognition Mathem...,Technology,High paying
2,3,Marketing Manager,Lead marketing campaigns and brand strategy de...,Creative Thinking Strategic Planning Communica...,Marketing,Average paying
3,4,UX Designer,Design user-friendly interfaces and improve us...,Creative Problem Solving Empathy Research Skil...,Technology,Average paying
4,5,Financial Analyst,Analyze financial data and prepare reports for...,Analytical Thinking Attention to Detail Mathem...,Finance,Average paying
5,6,Project Manager,Coordinate cross-functional teams to deliver p...,Leadership Organization Time Management Commun...,Business Services,Average paying



✅ All required columns present.


In [3]:
# Cell 2 — Clean and Combine Text Columns

import re

# Function to clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()

# Apply cleaning
for col in ["job_title", "Short_description", "Skills_required", "Industry"]:
    df[col] = df[col].apply(clean_text)

# Combine text fields
df["combined_text"] = (
    df["job_title"] + " " +
    df["Short_description"] + " " +
    df["Skills_required"] + " " +
    df["Industry"]
)

print("✅ Text cleaning and combination complete!")
print("\nSample combined text:")
print(df["combined_text"].head(5))


✅ Text cleaning and combination complete!

Sample combined text:
0    software engineer develop and maintain web app...
1    data scientist analyze large datasets to extra...
2    marketing manager lead marketing campaigns and...
3    ux designer design userfriendly interfaces and...
4    financial analyst analyze financial data and p...
Name: combined_text, dtype: object


In [4]:
# Cell 3 — Generate Embeddings and Save Model Data

!pip install -q sentence-transformers

import numpy as np
from sentence_transformers import SentenceTransformer

# Load transformer model
model_st = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
job_embeddings = model_st.encode(df["combined_text"].tolist(), show_progress_bar=True)

# Save all data for inference
df.to_csv("career_data_processed.csv", index=False)
np.save("career_embeddings.npy", job_embeddings)
model_st.save("career_bert_model")

print("✅ Embeddings generated and files saved successfully!")
print("Embedding shape:", job_embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

✅ Embeddings generated and files saved successfully!
Embedding shape: (970, 384)


In [7]:
# Cell 4 — Inference: Recommend Careers Based on User Input (Final + Fixed)

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load saved data and model
df = pd.read_csv("career_data_processed.csv")
job_embeddings = np.load("career_embeddings.npy")
model_st = SentenceTransformer("career_bert_model")

# Function to recommend careers
def recommend_careers(user_input, top_n=5):
    # Encode user query
    user_embedding = model_st.encode(user_input, convert_to_tensor=True)

    # Compute cosine similarity
    similarities = util.cos_sim(user_embedding, job_embeddings)[0]

    # Sort and get top N results
    top_results = np.argsort(-similarities.cpu().numpy())[:top_n]

    print(f"\n🔍 Based on your input: {user_input}\n")
    for idx in top_results:
        print(f"🎯 Career: {df.iloc[idx]['job_title']}")
        print(f"💡 Description: {df.iloc[idx]['Short_description'][:150]}...")
        print(f"🧠 Key Skills: {df.iloc[idx]['Skills_required']}")
        print(f"🏭 Industry: {df.iloc[idx]['Industry']}")
        print(f"💰 Pay Grade: {df.iloc[idx]['Pay_grade']}")
        print("—" * 60)

# Example run
user_query = "I love programming, AI, and solving complex problems"
recommend_careers(user_query, top_n=5)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔍 Based on your input: I love programming, AI, and solving complex problems

🎯 Career: mathematician
💡 Description: solve mathematical problems and develop mathematical theories...
🧠 Key Skills: logical reasoning abstract thinking problem solving pattern recognition analytical thinking
🏭 Industry: science  research
💰 Pay Grade: High paying
————————————————————————————————————————————————————————————
🎯 Career: robotics engineer
💡 Description: design and program automated systems and robots...
🧠 Key Skills: programming mechanical design systems integration problem solving innovation
🏭 Industry: robotics
💰 Pay Grade: High paying
————————————————————————————————————————————————————————————
🎯 Career: telepathic interface developer
💡 Description: create brain to brain communication systems...
🧠 Key Skills: neuroscience programming innovation problem solving technical skills
🏭 Industry: science  research
💰 Pay Grade: High paying
————————————————————————————————————————————————————————————
🎯 

In [8]:
# ✅ Cell 4 — Final Inference Code (Skills + Interests + Skill Gap + Pay Scale)

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load processed data and model
df = pd.read_csv("career_data_processed.csv")
job_embeddings = np.load("career_embeddings.npy")
model_st = SentenceTransformer("career_bert_model")

# Function to recommend careers based on user input
def recommend_careers(user_skills, user_interests, top_n=5):
    # Combine user input
    user_text = f"{user_skills} {user_interests}"
    user_embedding = model_st.encode(user_text, convert_to_tensor=True)

    # Compute cosine similarity
    similarities = util.cos_sim(user_embedding, job_embeddings)[0]
    top_results = np.argsort(-similarities.cpu().numpy())[:top_n]

    print(f"\n🔍 Based on your Skills: {user_skills}")
    print(f"❤️ Interests: {user_interests}\n")

    for idx in top_results:
        job = df.iloc[idx]
        
        # Extract job details
        job_title = job['job_title']
        description = job['Short_description']
        required_skills = [s.lower().strip() for s in job['Skills_required'].split()]
        pay_grade = job['Pay_grade']

        # Calculate missing skills
        user_skill_list = [s.lower().strip() for s in user_skills.split()]
        missing_skills = [s for s in required_skills if s not in user_skill_list]

        print(f"🎯 Career Opportunity: {job_title}")
        print(f"💰 Pay Scale: {pay_grade}")
        print(f"💡 Description: {description[:150]}...")
        print(f"🧩 Skill Gap: {', '.join(missing_skills[:5]) if missing_skills else 'No major gaps ✅'}")
        print("—" * 60)

# Example usage
user_skills = "Python machine learning data analysis teamwork problem solving"
user_interests = "artificial intelligence technology research innovation"

recommend_careers(user_skills, user_interests, top_n=5)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔍 Based on your Skills: Python machine learning data analysis teamwork problem solving
❤️ Interests: artificial intelligence technology research innovation

🎯 Career Opportunity: principal data scientist
💰 Pay Scale: High paying
💡 Description: lead data science initiatives and advanced analytics projects...
🧩 Skill Gap: advanced, statistics, team, leadership, business
————————————————————————————————————————————————————————————
🎯 Career Opportunity: machine learning engineer
💰 Pay Scale: High paying
💡 Description: design and implement ml algorithms for production systems...
🧩 Skill Gap: tensorflow, statistics, cloud, platforms, engineering
————————————————————————————————————————————————————————————
🎯 Career Opportunity: artificial intelligence trainer
💰 Pay Scale: High paying
💡 Description: develop and train machine learning models...
🧩 Skill Gap: programming, science, algorithm, development
————————————————————————————————————————————————————————————
🎯 Career Opportunity: ai researc

In [9]:
# ✅ Cell 4 — Final Inference Code (Skills + Interests + Skill Gap + Pay Scale)

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load processed data and model
df = pd.read_csv("career_data_processed.csv")
job_embeddings = np.load("career_embeddings.npy")
model_st = SentenceTransformer("career_bert_model")

# Function to recommend careers based on user input
def recommend_careers(user_skills, user_interests, top_n=5):
    # Combine user input
    user_text = f"{user_skills} {user_interests}"
    user_embedding = model_st.encode(user_text, convert_to_tensor=True)

    # Compute cosine similarity
    similarities = util.cos_sim(user_embedding, job_embeddings)[0]
    top_results = np.argsort(-similarities.cpu().numpy())[:top_n]

    print(f"\n🔍 Based on your Skills: {user_skills}")
    print(f"❤️ Interests: {user_interests}\n")

    for idx in top_results:
        job = df.iloc[idx]
        
        # Extract job details
        job_title = job['job_title']
        description = job['Short_description']
        required_skills = [s.lower().strip() for s in job['Skills_required'].split()]
        pay_grade = job['Pay_grade']

        # Calculate missing skills
        user_skill_list = [s.lower().strip() for s in user_skills.split()]
        missing_skills = [s for s in required_skills if s not in user_skill_list]

        print(f"🎯 Career Opportunity: {job_title}")
        print(f"💰 Pay Scale: {pay_grade}")
        print(f"💡 Description: {description[:150]}...")
        print(f"🧩 Skill Gap: {', '.join(missing_skills[:5]) if missing_skills else 'No major gaps ✅'}")
        print("—" * 60)

# Example usage
user_skills = "social , oilitics"
user_interests = "i am interested in politics and local governance"

recommend_careers(user_skills, user_interests, top_n=5)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


🔍 Based on your Skills: social , oilitics
❤️ Interests: i am interested in politics and local governance

🎯 Career Opportunity: political scientist
💰 Pay Scale: High paying
💡 Description: study political systems and analyze political behavior...
🧩 Skill Gap: research, skills, analytical, thinking, critical
————————————————————————————————————————————————————————————
🎯 Career Opportunity: indigenous food sovereignty advocate
💰 Pay Scale: Average paying
💡 Description: promote traditional food systems and cultural preservation...
🧩 Skill Gap: cultural, awareness, advocacy, skills, communication
————————————————————————————————————————————————————————————
🎯 Career Opportunity: ecological economist
💰 Pay Scale: High paying
💡 Description: study the relationship between economic systems and environmental health...
🧩 Skill Gap: economics, environmental, science, research, skills
————————————————————————————————————————————————————————————
🎯 Career Opportunity: digital nomad tax advisor
💰 Pay 